In [ ]:
# ============================================================
# GOOGLE COLAB SETUP — run this cell first when using Colab
# ============================================================
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # 1. Mount Google Drive so your data is accessible
    from google.colab import drive
    drive.mount('/content/drive')

    # 2. Set REPO_PATH to wherever you stored (or will store) the repo on Drive.
    #    If the folder doesn't exist the repo is cloned there automatically.
    REPO_PATH = '/content/drive/MyDrive/Factor-Research'

    if not os.path.exists(REPO_PATH):
        print('Cloning repository to Google Drive...')
        os.system(f'git clone https://github.com/mbrennan5/Factor-Research.git {REPO_PATH}')
    else:
        print(f'Repository found at {REPO_PATH}')

    # 3. Install required packages.
    #    Most are already in Colab; only the non-standard ones need installing.
    print('Installing packages...')
    os.system('pip install -q lightgbm xgboost optuna plotly tqdm yfinance')

    # 4. Change to the notebooks directory so relative paths (../data/...) work.
    NOTEBOOKS_DIR = os.path.join(REPO_PATH, 'notebooks')
    os.chdir(NOTEBOOKS_DIR)
    print(f'Working directory set to: {os.getcwd()}')
else:
    print('Running locally — no Colab setup needed.')


# Alpha Factor Generation - Technical & Fundamental Factors

This notebook implements a **comprehensive alpha factor generation system**. It serves as the factor library for the project, creating a wide range of alpha factors from base financial data.

Generate **alpha factors** covering multiple categories:
-   **Classic Alphas**: Replicate well-known factors like the Alpha101 series.
-   **Liquidity Factors**: Capture trading activity and market depth.
-   **Volatility & Momentum**: Measure price fluctuation and trend strength.
-   **Advanced Technical**: Develop composite factors from multiple indicators.
-   **Correlation Factors**: Analyze relationships between different price/volume metrics.


**Methodology**

-   **Vectorized Operations**: Use `pandas` and `numpy` for high-performance, vectorized calculations.
-   **Technical Operators**: Implement a library of time-series (`ts_`) and cross-sectional (`rank`) operators.
-   **Factor Naming Convention**: Use descriptive names for easy identification and analysis.


In [ ]:
# === 1. Import Libraries ===

import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import warnings
from numpy.lib.stride_tricks import sliding_window_view
from scipy.stats import rankdata

warnings.filterwarnings('ignore')

print("📚 Libraries imported successfully.")


In [ ]:
# === 2. Core Technical Operator Functions ===

# --- Time-Series Operators ---

def ts_max(df, window=10):
    """
    Calculates the rolling maximum over a specified window.
    Equivalent to `ts_max(x, d)` in Alpha101.
    """
    return df.rolling(window).max()

def ts_min(df, window=10):
    """
    Calculates the rolling minimum over a specified window.
    Equivalent to `ts_min(x, d)` in Alpha101.
    """
    return df.rolling(window).min()

def ts_sum(df, window=10):
    """
    Calculates the rolling sum over a specified window.
    Equivalent to `ts_sum(x, d)` in Alpha101.
    """
    return df.rolling(window).sum()

def ts_std_dev(df, window=10):
    """
    Calculates the rolling standard deviation over a specified window.
    Equivalent to `stddev(x, d)` in Alpha101.
    """
    return df.rolling(window).std()

def ts_delta(df, period=1):
    """
    Calculates the difference between the current value and a past value.
    Equivalent to `delta(x, d)` in Alpha101.
    """
    return df.diff(period)

def delay(df, period=1):
    """
    Shifts the DataFrame to get past values.
    Equivalent to `delay(x, d)` in Alpha101.
    """
    return df.shift(period)

def ts_rank(df, window=10):
    """
    Calculates the rolling rank of each value.
    Returns the percentage rank (0 to 1) of the current value in the trailing window.
    Equivalent to `ts_rank(x, d)` in Alpha101.
    """
    return df.rolling(window).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)


# --- Cross-Sectional Operator ---

def rank(df):
    """
    Calculates the cross-sectional rank of each value for each day.
    Returns the percentage rank (0 to 1).
    Equivalent to `rank(x)` in Alpha101.
    """
    return df.rank(axis=1, pct=True)


# --- Fill NaN Helper ---

def fill_nan(df):
    """
    Fills NaN values with the cross-sectional mean of that day.
    """
    return df.apply(lambda x: x.fillna(x.mean()), axis=1)


print("✅ Core technical operator functions defined.")
print("   - Time-Series: ts_max, ts_min, ts_sum, ts_std_dev, ts_delta, delay, ts_rank")
print("   - Cross-Sectional: rank")


In [ ]:
# === 3. Advanced Technical Functions (Correlation & Covariance) ===

def correlation(x, y, window=10):
    """
    Calculates the rolling correlation between two DataFrames.
    Equivalent to `correlation(x, y, d)` in Alpha101.
    """
    return x.rolling(window).corr(y)

def covariance(x, y, window=10):
    """
    Calculates the rolling covariance between two DataFrames.
    Equivalent to `covariance(x, y, d)` in Alpha101.
    """
    return x.rolling(window).cov(y)

print("✅ Advanced technical functions defined: correlation, covariance.")


In [ ]:
# === 4. Load Base Data for Factor Calculation ===

print("--- Loading base data for factor generation ---")
base_data_path = '../data/processed/wide_data_preparation/'

def load_data(name, path=base_data_path):
    """Loads a single CSV file, setting the index to datetime."""
    file_path = os.path.join(path, f"{name}.csv")
    try:
        df = pd.read_csv(file_path, index_col=0)
        df.index = pd.to_datetime(df.index)
        print(f"✅ Successfully loaded '{name}' data. Shape: {df.shape}")
        return df
    except FileNotFoundError:
        print(f"❌ Error: '{name}.csv' not found at {file_path}. Please run Data Preparation first.")
        return None

# Load all required base datasets
close = load_data('daily_close_data')
open_price = load_data('daily_close_data').shift(1) # Note: Using shifted close for open
high = load_data('daily_close_data') # Placeholder, assuming high/low are not prepared separately
low = load_data('daily_close_data') # Placeholder
volume = load_data('daily_money_data') # Using money as a proxy for volume based on typical data
ret = load_data('daily_ret_data')
vwap = load_data('vwap_daily_data')
turnover = load_data('daily_turnover_data')

# Check if all data was loaded successfully
if all(df is not None for df in [close, open_price, high, low, volume, ret, vwap, turnover]):
    print("\n--- All base data loaded successfully. Ready for factor generation. ---")
    data_loaded_successfully = True
else:
    print("\n--- ❌ Critical data missing. Factor generation may fail. ---")
    data_loaded_successfully = False


In [ ]:
# === 5. Helper Function & Classic Alpha101 Factors ===

# --- Helper function to save factors ---
output_path = '../data/factors/obtained_features'
if not os.path.exists(output_path):
    os.makedirs(output_path)

def save_factor(df, name):
    """Saves a factor DataFrame to the output directory."""
    if df is not None:
        file_path = os.path.join(output_path, f"{name}.csv")
        df.to_csv(file_path)
        print(f"   -> Saved {name}.csv")
    else:
        print(f"   -> Skipping {name} due to calculation error.")

if data_loaded_successfully:
    print("\n--- Generating Classic Alpha101 Factors ---")

    # Alpha#1: (rank(ts_argmax(signedpower((returns < 0) ? stddev(returns, 20) : close, 2.), 5)) - 0.5)
    # Simplified version: rank of return vs close volatility
    alpha1 = rank(close - ts_max(close, 20))
    save_factor(alpha1, 'alpha_001')

    # Alpha#2: (-1 * correlation(rank(delta(log(volume), 2)), rank(((close - open) / open)), 6))
    alpha2 = -1 * correlation(rank(ts_delta(volume.apply(np.log), 2)), rank((close - open_price) / open_price), 6)
    save_factor(alpha2, 'alpha_002')

    # Alpha#3: (-1 * correlation(rank(open), rank(volume), 10))
    alpha3 = -1 * correlation(rank(open_price), rank(volume), 10)
    save_factor(alpha3, 'alpha_003')

    # Alpha#4: (-1 * ts_rank(rank(low), 9))
    alpha4 = -1 * ts_rank(rank(low), 9)
    save_factor(alpha4, 'alpha_004')

    # Alpha#6: (-1 * correlation(open, volume, 10))
    alpha6 = -1 * correlation(open_price, volume, 10)
    save_factor(alpha6, 'alpha_006')

    # Alpha#7: (-1 * adv20 < volume) ? ((-1 * ts_rank(abs(delta(close, 7)), 60)) * sign(delta(close, 7))) : (-1 * 1)
    adv20 = ts_sum(volume, 20) / 20
    alpha7 = -1 * ts_rank(abs(ts_delta(close, 7)), 60) * np.sign(ts_delta(close, 7))
    alpha7[adv20 >= volume] = -1
    save_factor(alpha7, 'alpha_007')
    
    # Alpha#8: (-1 * rank(((sum(open, 5) * sum(returns, 5)) - delay((sum(open, 5) * sum(returns, 5)), 10))))
    alpha8 = -1 * rank(ts_sum(open_price, 5) * ts_sum(ret, 5) - delay(ts_sum(open_price, 5) * ts_sum(ret, 5), 10))
    save_factor(alpha8, 'alpha_008')
    
    # Alpha#9: ((0 < ts_min(delta(close, 1), 5)) ? delta(close, 1) : ((ts_max(delta(close, 1), 5) < 0) ? delta(close, 1) : (-1 * delta(close, 1))))
    delta_close = ts_delta(close, 1)
    cond_1 = ts_min(delta_close, 5) > 0
    cond_2 = ts_max(delta_close, 5) < 0
    alpha9 = -1 * delta_close
    alpha9[cond_1 | cond_2] = delta_close
    save_factor(alpha9, 'alpha_009')

    # Alpha#10: rank(((0 < ts_min(delta(close, 1), 4)) ? delta(close, 1) : ((ts_max(delta(close, 1), 4) < 0) ? delta(close, 1) : (-1 * delta(close, 1)))))
    delta_close_4 = ts_delta(close, 1)
    cond_1_4 = ts_min(delta_close_4, 4) > 0
    cond_2_4 = ts_max(delta_close_4, 4) < 0
    alpha10_base = -1 * delta_close_4
    alpha10_base[cond_1_4 | cond_2_4] = delta_close_4
    alpha10 = rank(alpha10_base)
    save_factor(alpha10, 'alpha_010')

else:
    print("--- ❌ Skipping Classic Alpha generation due to missing base data. ---")


In [ ]:
# === 6. Liquidity Factors ===

if data_loaded_successfully:
    print("\n--- Generating Liquidity Factors ---")
    
    # Factor: Volume momentum
    alpha_liq1 = rank(ts_delta(volume, 5))
    save_factor(alpha_liq1, 'alpha_liq_01')

    # Factor: 20-day average volume
    alpha_liq2 = rank(ts_sum(volume, 20) / 20)
    save_factor(alpha_liq2, 'alpha_liq_02')
    
    # Factor: Turnover rank
    alpha_liq3 = rank(turnover)
    save_factor(alpha_liq3, 'alpha_liq_03')

    # Factor: Turnover momentum
    alpha_liq4 = rank(ts_delta(turnover, 5))
    save_factor(alpha_liq4, 'alpha_liq_04')

else:
    print("--- ❌ Skipping Liquidity Factor generation due to missing base data. ---")


In [ ]:
# === 7. Volatility and Momentum Factors ===

if data_loaded_successfully:
    print("\n--- Generating Volatility and Momentum Factors ---")

    # Factor: 20-day return volatility (standard deviation)
    alpha_vol1 = rank(ts_std_dev(ret, 20))
    save_factor(alpha_vol1, 'alpha_vol_01')

    # Factor: 10-day price momentum
    alpha_mom1 = rank(close / delay(close, 10) - 1)
    save_factor(alpha_mom1, 'alpha_mom_01')

    # Factor: 20-day price momentum
    alpha_mom2 = rank(close / delay(close, 20) - 1)
    save_factor(alpha_mom2, 'alpha_mom_02')

    # Factor: Max price change over 10 days
    alpha_vol2 = rank(ts_max(high, 10) / ts_min(low, 10) - 1)
    save_factor(alpha_vol2, 'alpha_vol_02')

else:
    print("--- ❌ Skipping Volatility & Momentum Factor generation due to missing base data. ---")


In [ ]:
# === 8. Advanced Technical and Composite Factors ===

if data_loaded_successfully:
    print("\n--- Generating Advanced Technical and Composite Factors ---")

    # Factor: Close price's position within its 20-day high-low range
    alpha_tech1 = rank((close - ts_min(low, 20)) / (ts_max(high, 20) - ts_min(low, 20)))
    save_factor(alpha_tech1, 'alpha_tech_01')

    # Factor: Correlation between returns and volume over 5 days
    alpha_tech2 = rank(correlation(ret, volume, 5))
    save_factor(alpha_tech2, 'alpha_tech_02')
    
    # Factor: Price change relative to VWAP
    alpha_tech3 = rank((close - vwap) / close)
    save_factor(alpha_tech3, 'alpha_tech_03')

    # Factor: Moving average convergence/divergence (MACD) signal
    macd = (close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean())
    signal = macd.ewm(span=9, adjust=False).mean()
    alpha_tech4 = rank(macd - signal)
    save_factor(alpha_tech4, 'alpha_tech_04')
    
else:
    print("--- ❌ Skipping Advanced Factor generation due to missing base data. ---")


In [ ]:
# === 9. Turnover and Correlation Factors (Additional Examples) ===

if data_loaded_successfully:
    print("\n--- Generating Turnover and Correlation Factors ---")

    # Factor: Rank of 5-day mean turnover
    alpha_turn1 = rank(ts_sum(turnover, 5) / 5)
    save_factor(alpha_turn1, 'alpha_turn_01')

    # Factor: Correlation between rank of turnover and rank of volume
    alpha_corr1 = rank(correlation(rank(turnover), rank(volume), 5))
    save_factor(alpha_corr1, 'alpha_corr_01')
    
    # Factor: Correlation between close and vwap
    alpha_corr2 = rank(correlation(close, vwap, 10))
    save_factor(alpha_corr2, 'alpha_corr_02')

else:
    print("--- ❌ Skipping Turnover & Correlation Factor generation due to missing base data. ---")


In [ ]:
# === 10. Final Summary and Validation ===

print("\n" + "="*80)
print("✅ ALPHA FACTOR GENERATION COMPLETE")
print("="*80)

if data_loaded_successfully:
    # List all generated factor files
    generated_files = [f for f in os.listdir(output_path) if f.startswith('alpha_') and f.endswith('.csv')]
    
    print(f"\nTotal factors generated: {len(generated_files)}")
    print(f"All factors have been saved to: {output_path}")
    
    if generated_files:
        print("\n--- Sample of Generated Factors ---")
        for f in generated_files[:5]:
            print(f"  - {f}")
        
        # --- Validation Step ---
        # Load one factor to check its integrity
        try:
            sample_factor = pd.read_csv(os.path.join(output_path, generated_files[0]), index_col=0)
            print("\n--- Validation Check ---")
            print(f"Successfully loaded sample factor '{generated_files[0]}'")
            print(f"Shape: {sample_factor.shape}")
            print(f"Date Range: {sample_factor.index.min()} to {sample_factor.index.max()}")
            print("Validation successful. Factors appear to be correctly generated.")
        except Exception as e:
            print(f"\n--- ❌ Validation Failed ---")
            print(f"Could not read back sample factor. Error: {e}")
    else:
        print("\nNo factors were generated.")

else:
    print("\nNo factors were generated due to missing base data.")
    print("Please ensure 'Data Preparation.ipynb' has been run successfully.")

print("\n--- Next Step: Proceed to 'Alpha_Factor_Selection.ipynb' ---")
